In [ ]:
 #Paper where the LTV MPC is comming from https://arxiv.org/pdf/2002.06835
# Bellow is the building matrices for cost function and constraints
import numpy as np

# Horizon length step look forward
N=4
# Number of states
n_x=7
# Number of control inputs
n_u=1

# Penalty matrix for input
R_k=np.identity(n_u)
R=np.kron(np.identity(N-1),R_k)

# Cross penalty matrix for state and input
S_k=np.zeros((n_x,n_u))
S=np.kron(np.eye(N-1,k=1),S_k)

# Penalty matrix for the state
Q_k=np.identity(n_x)
Q=np.kron(np.eye(N-1,k=0),Q_k)
# Discrete system matrix A
A_k=np.zeros((n_x,n_x))
rng = np.random.default_rng()
A_k=rng.uniform(low=-1, high=1, size=(n_x,n_x))
A=np.kron(np.eye(N-1,k=-1),-A_k)+np.kron(np.eye(N-1),np.identity(n_x))


# Discrete system input matrix B
B_k=np.zeros((n_x,n_u))
B_k=rng.uniform(low=-1, high=1, size=(n_x,n_u))
B=np.kron(np.eye(N-1,k=0),B_k)

x_0=np.zeros((n_x,1))


r_k=np.zeros((n_u,1))
r_k=np.ones((n_u,1))

r=np.concatenate((S_k.T@x_0+r_k,np.kron(np.ones((N-2,1)),r_k)),axis=0)
q_k=np.zeros((n_x,1))
q=np.kron(np.ones((N-1,1)),q_k)

# Disturbance matrix
w_k=np.zeros((n_x,1))
w=np.concatenate((A_k@x_0+w_k,np.kron(np.ones((N-2,1)),w_k)),axis=0)

# Upper and lower bounds for control action
x_k=np.array([[1],[1],[1],[1],[1],[1],[1]])
x_lower_bound=np.kron(np.ones((N-1,1)),-x_k)
x_upper_bound=np.kron(np.ones((N-1,1)),x_k)



In [9]:
import cvxpy as cp
import numpy as np

x_1 = cp.Variable(N-1)
x_2 = cp.Variable(N-1)
x_3 = cp.Variable(N-1)
x_4 = cp.Variable(N-1)
x_5 = cp.Variable(N-1)
x_6 = cp.Variable(N-1)
x_7 = cp.Variable(N-1)
u=cp.Variable((N-1)*n_u)
x = cp.hstack([
    cp.hstack([x_1[k], x_2[k], x_3[k], x_4[k], x_5[k], x_6[k], x_7[k]])
    for k in range(N-1)
])

# Hessian matrix
H = np.block([
    [R, S.T],
    [S, Q]
])

# Decision vector
z = cp.hstack([u,x])

# Linear coefficient
d = np.concatenate((r, q), axis=0)

# Constant
c = 0.5 * x_0.T @ Q_k @ x_0 + q_k.T @ x_0

# Objective
cost = 0.5 * cp.quad_form(z, H) + d.T @ z + c.item()

# Constraints
constraints = [
    A @ x == B@u + w.flatten(),
    x_lower_bound.flatten()<=x,
    x<=x_upper_bound.flatten()
    #u_lower_bound.flatten() <= u,
    #u <= u_upper_bound.flatten()
]

problem = cp.Problem(cp.Minimize(cost), constraints)

problem.solve()

print("status:", problem.status)
print("optimal value:", problem.value)
print("u* =", u.value)
print("x* =", x.value)


status: optimal
optimal value: -0.32589091745802823
u* = [-0.10755986 -0.14389628 -0.4003257 ]
x* = [ 0.02273465 -0.08739388  0.0093608   0.05845563 -0.0395549  -0.01323354
  0.0834664  -0.06236664 -0.0798215   0.16194023  0.04020831 -0.16033225
 -0.07002512  0.07402664 -0.07635749 -0.34072914  0.280701    0.11730178
 -0.31448502 -0.1725085   0.14679378]
